# 🛠️ Automated Customer Data Cleaning & Standardization Project
**Author:** Ripan Nurpaujan | Data Analyst
**Tools:** Python (Pandas, Regex)

### 📋 Project Objective
To automate the cleaning process of a "dirty" customer contact dataset. The raw data contains inconsistent formatting, duplicate entries, and missing values that hinder marketing analysis.

**Key Technical Goals:**
1.  **Remove Duplicates:** Ensure unique customer records based on phone numbers.
2.  **Standardize Formats:** Clean inconsistent phone number patterns.
3.  **Data Imputation:** Handle `NaN` and null values for data integrity.

## 1. Data Acquisition & Inspection
Loading the raw dataset to identify inconsistencies and format errors.

In [1]:
import kagglehub

path = kagglehub.dataset_download("singhshivani0696/customer-call-list")

print("Path to dataset files:", path)

C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\ASUS\.cache\kagglehub\datasets\singhshivani0696\customer-call-list\versions\1


In [2]:
import pandas as pd
import numpy as np
import os

file_path = os.path.join(path, "Customer Call List.xlsx")

df = pd.read_excel(file_path)
df

,CustomerID,First_Name,Last_Name,Phone_Number,Address,Paying Customer,Do_Not_Contact,Not_Useful_Column
0,1001,Frodo,Baggins,123-545-5421,"123 Shire Lane, Shire",Yes,No,True
1,1002,Abed,Nadir,123/643/9775,93 West Main Street,No,Yes,False
2,1003,Walter,/White,7066950392,298 Drugs Driveway,N,NaN,True
3,1004,Dwight,Schrute,123-543-2345,"980 Paper Avenue, Pennsylvania, 18503",Yes,Y,True
4,1005,Jon,Snow,876|678|3469,123 Dragons Road,Y,No,True
5,1006,Ron,Swanson,304-762-2467,768 City Parkway,Yes,Yes,True
6,1007,Jeff,Winger,NaN,1209 South Street,No,No,False
7,1008,Sherlock,Holmes,876|678|3469,98 Clue Drive,N,No,False
8,1009,Gandalf,NaN,N/a,123 Middle Earth,Yes,NaN,False
9,1010,Peter,Parker,123-545-5421,"25th Main Street, New York",Yes,No,True


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   CustomerID         21 non-null     int64 
 1   First_Name         21 non-null     object
 2   Last_Name          20 non-null     object
 3   Phone_Number       19 non-null     object
 4   Address            21 non-null     object
 5   Paying Customer    21 non-null     object
 6   Do_Not_Contact     17 non-null     object
 7   Not_Useful_Column  21 non-null     bool  
dtypes: bool(1), int64(1), object(6)
memory usage: 1.3+ KB


In [4]:
df.duplicated().sum()

1

## 2. Data Cleaning Process
Applying Python logic to transform the raw data into a usable format.

**Steps Taken:**
* **De-duplication:** Identified and removed duplicate rows based on the `Phone_Number` column to prevent redundant client contact.
* **Handling Null Values:** Imputed missing values in the `Do_Not_Contact` column (defaulting to "No") and replaced `NaN`/`N/a` with blank strings for cleaner visualization.
* **Normalization:** Resetting the index to maintain a structured sequence after row deletion.

In [5]:
# removing duplicates
df.drop_duplicates(inplace=True)


In [6]:
df.drop(columns='Not_Useful_Column', inplace=True)

In [7]:
# remove whitespace
df["First_Name"] = df["First_Name"].str.strip()
df["Last_Name"] = df["Last_Name"].str.strip("/._")
df[['First_Name','Last_Name']]

,First_Name,Last_Name
0,Frodo,Baggins
1,Abed,Nadir
2,Walter,White
3,Dwight,Schrute
4,Jon,Snow
5,Ron,Swanson
6,Jeff,Winger
7,Sherlock,Holmes
8,Gandalf,NaN
9,Peter,Parker


In [8]:
# remove character in phone number column
df['Phone_Number'] = df['Phone_Number'].astype(str)
df['Phone_Number'] = df['Phone_Number'].str.replace(r'\D', '', regex=True)
df['Phone_Number']


0     1235455421
1     1236439775
2     7066950392
3     1235432345
4     8766783469
5     3047622467
6               
7     8766783469
8               
9     1235455421
10              
11    7066950392
12    1235432345
13    8766783469
14    3047622467
15    1235455421
16    1236439775
17    7066950392
18              
19    8766783469
Name: Phone_Number, dtype: object

In [9]:
# reformating phone number
df['Phone_Number'] = df['Phone_Number'].apply(lambda x: x[0:3]+'-'+x[3:7]+'-'+x[7:])
df['Phone_Number']

0     123-5455-421
1     123-6439-775
2     706-6950-392
3     123-5432-345
4     876-6783-469
5     304-7622-467
6               --
7     876-6783-469
8               --
9     123-5455-421
10              --
11    706-6950-392
12    123-5432-345
13    876-6783-469
14    304-7622-467
15    123-5455-421
16    123-6439-775
17    706-6950-392
18              --
19    876-6783-469
Name: Phone_Number, dtype: object

In [10]:
df['Phone_Number'] = df['Phone_Number'].str.replace('--', '')
df['Phone_Number']

0     123-5455-421
1     123-6439-775
2     706-6950-392
3     123-5432-345
4     876-6783-469
5     304-7622-467
6                 
7     876-6783-469
8                 
9     123-5455-421
10                
11    706-6950-392
12    123-5432-345
13    876-6783-469
14    304-7622-467
15    123-5455-421
16    123-6439-775
17    706-6950-392
18                
19    876-6783-469
Name: Phone_Number, dtype: object

In [11]:
df[["Street_Address", "State", "Zip_Code"]] = df["Address"].str.split(',', expand=True)
df.drop(columns='Address', inplace=True)
df

,CustomerID,First_Name,Last_Name,Phone_Number,Paying Customer,Do_Not_Contact,Street_Address,State,Zip_Code
0,1001,Frodo,Baggins,123-5455-421,Yes,No,123 Shire Lane,Shire,None
1,1002,Abed,Nadir,123-6439-775,No,Yes,93 West Main Street,None,None
2,1003,Walter,White,706-6950-392,N,NaN,298 Drugs Driveway,None,None
3,1004,Dwight,Schrute,123-5432-345,Yes,Y,980 Paper Avenue,Pennsylvania,18503
4,1005,Jon,Snow,876-6783-469,Y,No,123 Dragons Road,None,None
5,1006,Ron,Swanson,304-7622-467,Yes,Yes,768 City Parkway,None,None
6,1007,Jeff,Winger,,No,No,1209 South Street,None,None
7,1008,Sherlock,Holmes,876-6783-469,N,No,98 Clue Drive,None,None
8,1009,Gandalf,NaN,,Yes,NaN,123 Middle Earth,None,None
9,1010,Peter,Parker,123-5455-421,Yes,No,25th Main Street,New York,None


In [12]:
# Reordering the columns
ls = list(df.columns.values)
df = df[ls[0:4] + ls[6:]+ls[4:6]]
df

,CustomerID,First_Name,Last_Name,Phone_Number,Street_Address,State,Zip_Code,Paying Customer,Do_Not_Contact
0,1001,Frodo,Baggins,123-5455-421,123 Shire Lane,Shire,None,Yes,No
1,1002,Abed,Nadir,123-6439-775,93 West Main Street,None,None,No,Yes
2,1003,Walter,White,706-6950-392,298 Drugs Driveway,None,None,N,NaN
3,1004,Dwight,Schrute,123-5432-345,980 Paper Avenue,Pennsylvania,18503,Yes,Y
4,1005,Jon,Snow,876-6783-469,123 Dragons Road,None,None,Y,No
5,1006,Ron,Swanson,304-7622-467,768 City Parkway,None,None,Yes,Yes
6,1007,Jeff,Winger,,1209 South Street,None,None,No,No
7,1008,Sherlock,Holmes,876-6783-469,98 Clue Drive,None,None,N,No
8,1009,Gandalf,NaN,,123 Middle Earth,None,None,Yes,NaN
9,1010,Peter,Parker,123-5455-421,25th Main Street,New York,None,Yes,No


In [ ]:
df['Paying Customer'] = df['Paying Customer'].replace({'Y': 'Yes', 'N': 'No'})
df['Do_Not_Contact'] = df['Do_Not_Contact'].replace({'Y': 'Yes', 'N': 'No'})
df

,CustomerID,First_Name,Last_Name,Phone_Number,Street_Address,State,Zip_Code,Paying Customer,Do_Not_Contact
0,1001,Frodo,Baggins,123-5455-421,123 Shire Lane,Shire,None,Yes,No
1,1002,Abed,Nadir,123-6439-775,93 West Main Street,None,None,No,Yes
2,1003,Walter,White,706-6950-392,298 Drugs Driveway,None,None,No,NaN
3,1004,Dwight,Schrute,123-5432-345,980 Paper Avenue,Pennsylvania,18503,Yes,Yes
4,1005,Jon,Snow,876-6783-469,123 Dragons Road,None,None,Yes,No
5,1006,Ron,Swanson,304-7622-467,768 City Parkway,None,None,Yes,Yes
6,1007,Jeff,Winger,,1209 South Street,None,None,No,No
7,1008,Sherlock,Holmes,876-6783-469,98 Clue Drive,None,None,No,No
8,1009,Gandalf,NaN,,123 Middle Earth,None,None,Yes,NaN
9,1010,Peter,Parker,123-5455-421,25th Main Street,New York,None,Yes,No


In [14]:
# drop Do_not_contact (Yes) and phone_number (missing value)
df = df[df['Do_Not_Contact'] != 'Yes']
df = df[df['Phone_Number'] != '']
df = df.reset_index(drop=True)
df


,CustomerID,First_Name,Last_Name,Phone_Number,Street_Address,State,Zip_Code,Paying Customer,Do_Not_Contact
0,1001,Frodo,Baggins,123-5455-421,123 Shire Lane,Shire,None,Yes,No
1,1003,Walter,White,706-6950-392,298 Drugs Driveway,None,None,No,NaN
2,1005,Jon,Snow,876-6783-469,123 Dragons Road,None,None,Yes,No
3,1008,Sherlock,Holmes,876-6783-469,98 Clue Drive,None,None,No,No
4,1010,Peter,Parker,123-5455-421,25th Main Street,New York,None,Yes,No
5,1012,Harry,Potter,706-6950-392,2394 Hogwarts Avenue,None,None,Yes,NaN
6,1013,Don,Draper,123-5432-345,2039 Main Street,None,None,Yes,No
7,1014,Leslie,Knope,876-6783-469,343 City Parkway,None,None,Yes,No
8,1015,Toby,Flenderson,304-7622-467,214 HR Avenue,None,None,No,No
9,1016,Ron,Weasley,123-5455-421,2395 Hogwarts Avenue,None,None,No,No


In [15]:
df["Do_Not_Contact"] = df["Do_Not_Contact"].replace('', 'No')
df = df.reset_index(drop=True)
df

,CustomerID,First_Name,Last_Name,Phone_Number,Street_Address,State,Zip_Code,Paying Customer,Do_Not_Contact
0,1001,Frodo,Baggins,123-5455-421,123 Shire Lane,Shire,None,Yes,No
1,1003,Walter,White,706-6950-392,298 Drugs Driveway,None,None,No,NaN
2,1005,Jon,Snow,876-6783-469,123 Dragons Road,None,None,Yes,No
3,1008,Sherlock,Holmes,876-6783-469,98 Clue Drive,None,None,No,No
4,1010,Peter,Parker,123-5455-421,25th Main Street,New York,None,Yes,No
5,1012,Harry,Potter,706-6950-392,2394 Hogwarts Avenue,None,None,Yes,NaN
6,1013,Don,Draper,123-5432-345,2039 Main Street,None,None,Yes,No
7,1014,Leslie,Knope,876-6783-469,343 City Parkway,None,None,Yes,No
8,1015,Toby,Flenderson,304-7622-467,214 HR Avenue,None,None,No,No
9,1016,Ron,Weasley,123-5455-421,2395 Hogwarts Avenue,None,None,No,No


In [16]:
df["Do_Not_Contact"] = df["Do_Not_Contact"].fillna("No")
df = df.drop_duplicates(subset="Phone_Number", keep="first")
df = df.reset_index(drop=True)
df

,CustomerID,First_Name,Last_Name,Phone_Number,Street_Address,State,Zip_Code,Paying Customer,Do_Not_Contact
0,1001,Frodo,Baggins,123-5455-421,123 Shire Lane,Shire,None,Yes,No
1,1003,Walter,White,706-6950-392,298 Drugs Driveway,None,None,No,No
2,1005,Jon,Snow,876-6783-469,123 Dragons Road,None,None,Yes,No
3,1013,Don,Draper,123-5432-345,2039 Main Street,None,None,Yes,No
4,1015,Toby,Flenderson,304-7622-467,214 HR Avenue,None,None,No,No
5,1017,Michael,Scott,123-6439-775,121 Paper Avenue,Pennsylvania,None,Yes,No


## 3. Final Output & Export
The data is now clean, consistent, and duplicate-free. The final dataset is exported to CSV/Excel for further analysis or CRM integration.

In [17]:
df.to_excel("Customer_Call_List_Cleaned.xlsx", index=False)
df

,CustomerID,First_Name,Last_Name,Phone_Number,Street_Address,State,Zip_Code,Paying Customer,Do_Not_Contact
0,1001,Frodo,Baggins,123-5455-421,123 Shire Lane,Shire,None,Yes,No
1,1003,Walter,White,706-6950-392,298 Drugs Driveway,None,None,No,No
2,1005,Jon,Snow,876-6783-469,123 Dragons Road,None,None,Yes,No
3,1013,Don,Draper,123-5432-345,2039 Main Street,None,None,Yes,No
4,1015,Toby,Flenderson,304-7622-467,214 HR Avenue,None,None,No,No
5,1017,Michael,Scott,123-6439-775,121 Paper Avenue,Pennsylvania,None,Yes,No
